# `TodoListMiddleware`

Middleware that adds structured planning and task-tracking capabilities to an agent.

It registers a `write_todos` tool, stores the current todo list in agent state, injects instructions explaining when and how to use the tool, and prevents multiple parallel todo-list updates in a single model turn.

- Bases: `AgentMiddleware[PlanningState[ResponseT], ContextT, ResponseT]`
- State schema: `PlanningState`

## Constructor

```python
TodoListMiddleware(
    *,
    system_prompt: str = WRITE_TODOS_SYSTEM_PROMPT,
    tool_description: str = WRITE_TODOS_TOOL_DESCRIPTION
)
```

## Parameters

* `system_prompt` — Instructions appended to the model's system message.
  * Default: `WRITE_TODOS_SYSTEM_PROMPT`
  * Guides the agent on when to create, update, and complete todo items.
  * Explains that `write_todos` must not be called multiple times in parallel.
  * Reminds the agent to provide the actual final answer after its last todo update.

* `tool_description` — Description assigned to the registered `write_todos` tool.
  * Default: `WRITE_TODOS_TOOL_DESCRIPTION`
  * Explains when the tool is useful and when it should be skipped.
  * Defines todo statuses and task-completion rules.

## Attributes

* `state_schema` — Set to `PlanningState`.
* `system_prompt` — System prompt appended before every model call.
* `tool_description` — Description used for the dynamically created tool.
* `tools` — One-element list containing the `write_todos` `StructuredTool`.

The registered tool is created using:

```python
StructuredTool.from_function(
    name="write_todos",
    description=tool_description,
    func=_write_todos,
    coroutine=_awrite_todos,
    args_schema=WriteTodosInput,
    infer_schema=False,
)
```

# `Todo`

Typed dictionary representing one task.

```python
class Todo(TypedDict):
    content: str
    status: Literal[
        "pending",
        "in_progress",
        "completed"
    ]
```

## Fields

* `content` — Description of the task.
* `status` — Current task state.

Supported statuses:

| Status | Meaning |
|---|---|
| `pending` | Task has not started |
| `in_progress` | Task is currently being worked on |
| `completed` | Task has been fully finished |

## Example

```python
todo: Todo = {
    "content": "Inspect the project structure",
    "status": "in_progress",
}
```

# `PlanningState`

Agent-state schema used by the middleware.

```python
class PlanningState(AgentState[ResponseT]):
    todos: Annotated[
        NotRequired[list[Todo]],
        OmitFromInput
    ]
```

## Field

* `todos` — Current complete todo list.
  * Optional in the state.
  * Excluded from the normal external agent input schema through `OmitFromInput`.
  * Replaced whenever the `write_todos` tool runs.

The field is intended to be read from the agent result:

```python
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Help me refactor this codebase.",
            }
        ]
    }
)

print(result["todos"])
```

# `WriteTodosInput`

Pydantic input schema used by the dynamically registered tool.

- Bases: `BaseModel`

```python
class WriteTodosInput(BaseModel):
    todos: list[Todo]
```

## Field

* `todos` — The complete todo list that should replace the currently stored list.

The tool does not append one item automatically. Every call supplies the complete desired state of the list.

# `write_todos`

Module-level tool for creating or replacing the current todo list.

```python
write_todos(
    todos: list[Todo],
    tool_call_id: Annotated[
        str,
        InjectedToolCallId
    ]
) -> Command[Any]
```

## Parameters

* `todos` — Complete updated todo list.
* `tool_call_id` — Current tool-call ID injected by LangChain.

## Returns

A LangGraph `Command` containing:

```python
Command(
    update={
        "todos": todos,
        "messages": [
            ToolMessage(
                f"Updated todo list to {todos}",
                tool_call_id=tool_call_id,
            )
        ],
    }
)
```

The command performs two updates:

1. Replaces `state["todos"]`.
2. Adds a confirmation `ToolMessage` associated with the original tool call.

# Internal Tool Functions

## `_write_todos`

Synchronous function used by the middleware-created `StructuredTool`.

```python
_write_todos(
    runtime: ToolRuntime[
        ContextT,
        PlanningState[ResponseT]
    ],
    todos: list[Todo]
) -> Command[Any]
```

It reads the tool-call ID from:

```python
runtime.tool_call_id
```

and returns:

```python
Command(
    update={
        "todos": todos,
        "messages": [
            ToolMessage(
                f"Updated todo list to {todos}",
                tool_call_id=runtime.tool_call_id,
            )
        ],
    }
)
```

## `_awrite_todos`

Asynchronous wrapper around `_write_todos`.

```python
async def _awrite_todos(
    runtime: ToolRuntime[
        ContextT,
        PlanningState[ResponseT]
    ],
    todos: list[Todo]
) -> Command[Any]
```

It delegates directly to the synchronous implementation:

```python
return _write_todos(runtime, todos)
```

# Methods

## 1. `wrap_model_call`

Appends the todo-management instructions to the system message before a synchronous model call.

```python
wrap_model_call(
    self,
    request: ModelRequest[ContextT],
    handler: Callable[
        [ModelRequest[ContextT]],
        ModelResponse[ResponseT]
    ]
) -> ModelResponse[ResponseT] | AIMessage
```

## Behaviour

When the request already contains a system message, its existing content blocks are preserved and a new text block is appended:

```python
new_system_content = [
    *request.system_message.content_blocks,
    {
        "type": "text",
        "text": f"\n\n{self.system_prompt}",
    },
]
```

When no system message exists, a new one is created:

```python
new_system_content = [
    {
        "type": "text",
        "text": self.system_prompt,
    }
]
```

The modified request is then executed using:

```python
handler(
    request.override(
        system_message=new_system_message
    )
)
```

## 2. `awrap_model_call`

Asynchronous version of `wrap_model_call`.

```python
async def awrap_model_call(
    self,
    request: ModelRequest[ContextT],
    handler: Callable[
        [ModelRequest[ContextT]],
        Awaitable[ModelResponse[ResponseT]]
    ]
) -> ModelResponse[ResponseT] | AIMessage
```

It applies the same system-message transformation and awaits the handler:

```python
return await handler(
    request.override(
        system_message=new_system_message
    )
)
```

## 3. `after_model`

Checks whether the latest model response contains multiple parallel `write_todos` calls.

```python
after_model(
    self,
    state: PlanningState[ResponseT],
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

## Behaviour

The method:

1. Reads `state["messages"]`.
2. Searches backward for the latest `AIMessage`.
3. Reads its tool calls.
4. Filters calls whose name is `"write_todos"`.
5. Returns errors when more than one such call exists.
6. Returns `None` otherwise.

Parallel calls are rejected because every call replaces the whole todo list, making precedence ambiguous.

For every conflicting call, the middleware creates:

```python
ToolMessage(
    content=(
        "Error: The `write_todos` tool should never be "
        "called multiple times in parallel. Please call "
        "it only once per model invocation to update the "
        "todo list."
    ),
    tool_call_id=tool_call["id"],
    status="error",
)
```

The returned state update is:

```python
{
    "messages": error_messages
}
```

The original tool calls remain in the `AIMessage`; the generated error messages provide results for those calls.

Returns `None` when:

* The message list is empty.
* No `AIMessage` is present.
* The latest `AIMessage` has no tool calls.
* Zero or one `write_todos` call is present.

## 4. `aafter_model`

Asynchronous version of `after_model`.

```python
async def aafter_model(
    self,
    state: PlanningState[ResponseT],
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

It delegates directly to:

```python
return self.after_model(state, runtime)
```

# Tool Usage Guidance

The default tool description recommends using the todo list for:

1. Tasks requiring three or more distinct steps.
2. Non-trivial work needing careful planning.
3. Requests where the user explicitly asks for a todo list.
4. Requests containing several separate tasks.
5. Work where later steps may change after earlier results.

It recommends skipping the tool for:

1. One straightforward task.
2. Trivial work where tracking adds no value.
3. Work completed in fewer than three simple steps.
4. Purely conversational or informational questions.

# Task-Management Rules

The default prompts instruct the model to:

* Mark the first active task as `in_progress`.
* Keep at least one task `in_progress` while unfinished work remains.
* Mark a task `completed` immediately after it is fully finished.
* Avoid batching several completion updates together.
* Finish current tasks before starting dependent tasks.
* Add newly discovered follow-up work.
* Remove tasks that are no longer relevant.
* Keep blocked or incomplete work as `in_progress`.
* Create a new task describing a blocker when required.
* Avoid changing previously completed tasks.
* Never call `write_todos` multiple times in parallel.

A todo should not be marked `completed` when:

* Errors remain unresolved.
* Work is only partially finished.
* A blocker prevents completion.
* Required resources or dependencies are unavailable.
* Expected quality requirements have not been met.

# Final-Answer Rule

Updating the todo list is not considered the final response to the user.

The default prompts instruct the agent to:

1. Complete its last `write_todos` call.
2. Send a later assistant message containing the actual requested result.
3. Start that message with the substantive answer rather than a completion announcement.

Example flow:

```text
Assistant -> write_todos(completed list)
Tool      -> Updated todo list ...
Assistant -> Actual summary, computation, code, or analysis
```

# Todo Update Flow

```text
Model decides a structured plan is helpful
        |
        v
Call write_todos once
        |
        v
Replace state["todos"] with supplied list
        |
        v
Add confirmation ToolMessage
        |
        v
Continue the task
        |
        v
Call write_todos again when statuses change
        |
        v
Provide the actual user-facing answer after final update
```

# Parallel-Call Protection

```text
Model produces tool calls
        |
        v
Find write_todos calls in latest AIMessage
        |
        v
How many are present?
        |
  0 or 1 +--> Continue normally
        |
    More than 1
        |
        v
Return one error ToolMessage per conflicting call
```

The restriction applies per model invocation. The tool may still be called again in a later model turn to revise the list.

# Constants

## `WRITE_TODOS_TOOL_DESCRIPTION`

Default detailed tool description.

It explains:

* When to use the tool.
* When not to use it.
* The meaning of each task status.
* How to update tasks during execution.
* When a task may be marked complete.
* How to handle blockers.
* That the final todo update is not itself the user-facing answer.

## `WRITE_TODOS_SYSTEM_PROMPT`

Default system prompt appended before model execution.

It emphasizes:

* Using todos for complex objectives.
* Avoiding todos for simple tasks.
* Updating completed items immediately.
* Revising the list as new information appears.
* Never calling `write_todos` in parallel.
* Sending the requested result after the last todo-tool call.

# Examples

## Basic Usage

```python
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware

agent = create_agent(
    model="openai:gpt-5.5",
    middleware=[
        TodoListMiddleware()
    ],
)
```

The agent now has:

```text
Tool:  write_todos
State: todos
```

## Read the Final Todo State

```python
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Analyse this repository and suggest improvements.",
            }
        ]
    }
)

print(result.get("todos", []))
```

## Custom System Prompt

```python
middleware = TodoListMiddleware(
    system_prompt="""
Use write_todos only for development tasks that require
multiple dependent steps. Keep descriptions concise.
"""
)
```

## Custom Tool Description

```python
middleware = TodoListMiddleware(
    tool_description="""
Replace the current project task list.

Each item must contain:
- content
- status: pending, in_progress, or completed
"""
)
```

## Example Tool Input

```python
{
    "todos": [
        {
            "content": "Inspect the existing implementation",
            "status": "completed",
        },
        {
            "content": "Identify edge cases",
            "status": "in_progress",
        },
        {
            "content": "Write the final recommendation",
            "status": "pending",
        },
    ]
}
```

## Updating the List

A later call supplies the complete replacement list:

```python
{
    "todos": [
        {
            "content": "Inspect the existing implementation",
            "status": "completed",
        },
        {
            "content": "Identify edge cases",
            "status": "completed",
        },
        {
            "content": "Write the final recommendation",
            "status": "in_progress",
        },
    ]
}
```

# Source

This reference follows the pinned LangChain source:

```text
libs/langchain_v1/langchain/agents/middleware/todo.py
Commit: 42f8f79293cfb7589e5bc1d74a8ae4dfd0bf15e3
```